In [2]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(),'..'))
import CIBUSmod as cm

root: /home/niceri/Pythoncode/CIBUSmod/notebooks/../CIBUSmod/..
input_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/input
temp_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
export_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/exported_results


# Soil carbon and climate impact calculations
The code in this notebook calculates and manages C flows to and from soils as well as climate impacts.

In this section the `xarray` package is being heavily used for its excellent handling of multidimensional data.
It may take some time to get used to how to use it. For that reason, below is a cell with tips on how to learn using its functionalities.

For the functions and methods in the `SoilData` class to work the `xarray` package must be installed (pip install xarray)

In [4]:
FAI_soil = cm.SoilData.load_instance_state('FAI_soil')
FAI_soil.load_inventory()

The following dataset variables have been set:
 input_inventory
The following dataframe variables have been set:
 input_df
 ss_input_df
The following dataset variables have been set:
 soc_inventory
The following dataframe variables have been set:
 soc_ha_df
 soc_sko_df
The following dataset variables have been set:
 historic_inventory
The following dataframe variables have been set:
 historic_ha_df
 historic_sko_df
No dataset variables have been loaded.
No dataframe variables have been loaded.


The following code block can be used to check which variables have currently been set.
Both public and private attributes are shown to facilitate quick troubleshooting. 

In [5]:
FAI_soil.check_attributes_status('public')#, FAI_soil.check_attributes_status('private') 

('The following attributes are set for public variables',
 {'co2_fluxes': ('NoneType', False),
  'historic_ha_df': ('DataFrame', True),
  'historic_inventory': ('Dataset', True),
  'historic_sko_df': ('DataFrame', True),
  'input_df': ('DataFrame', True),
  'input_inventory': ('Dataset', True),
  'name': ('str', True),
  'scenario': ('str', True),
  'soc_ha_df': ('DataFrame', True),
  'soc_inventory': ('Dataset', True),
  'soc_sko_df': ('DataFrame', True),
  'ss_input_df': ('DataFrame', True),
  'startyear': ('NoneType', False)})

In [ ]:
# The commands in this cell saves the current state of the SoilData instance
FAI_soil.save_inventory()
FAI_soil.save_instance_state()

In [ ]:
FAI_soil = cm.SoilData.load_instance_state('FAI_soil')
FAI_soil.load_inventory()

## Calculate the carbon inputs for each input in the input_df
The `calc_scn_inputs` method calculates the C input for each fraction of `input_df` and sets the  following variable attributes:
- `startyear`: defined as the first year of the timeseries in `input_df`.
- `input_inventory`: xarray dataset with `scn, crop, prod_system, region, input_year` as coordinates, including all original input and the C input per ha as well as per SKO for all fractions.
- `ss_input_df.columns`: dataframe containing the SOC inputs to use for the spinup modelling, corresponding to the C inputs for all fraction in the `startyear`, also both per ha and per SKO. 

**Note:** the column or index level name `year` in the `input_df` will be renamed to `input_year` to distinguish it from other time coordinates calculated with ICBM and subsequent temperature response functions.

In [ ]:
FAI_soil.calc_scn_inputs(verbose=False)

## Calculate the SOC timeseries
The `calc_soc_timeseries` currently calculates the soc timeseries for each individual yearly input in the `input_inventory`. This is a memory intensive operation. It may cause the jupyter instance to crash if memory allocation is not sufficiently large.

**TODO:** Redefine function to operate on fractions of the database to reduce memory load.

In [ ]:
FAI_soil.calc_soc_timeseries(verbose=True)

### Calculate the historic SOC timeseries
The `calc_historic_soc_timeseries` calculates the soc timeseries for each individual SS input in the `ss_input_df`. The inputs all take place in the year 2020 (and only this year), making the computation much lighter and faster then the previous ones.


In [ ]:
FAI_soil.calc_historic_soc_timeseries(verbose=True)

## Calculate and assign the tot SOC and CO2 fluxes
When both future and historic SOC timeseries have been calculated for a scenario the following methods have to be run to calculate the total SOC and CO2 fluxes for each year.
Both will be added as new data arrays to the existing `soc_inventory` dataset

In [ ]:
FAI_soil.add_total_soc()
FAI_soil.add_co2_flux()

## Saving and loading datasets
The `SoilData` class has instance methods to save and read saved datasets to avoid having to rerun the computations between sessions. These are **NOT** automatic, but have to be invoked by the user.

The `save_inventory` and `load_inventory` methods can be called with three optional strings:

- `inputs` saves and loads the input inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The  netcdf file is named `<scenario_name>_input_ds.nc`
- `soc` saves and loads the soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_soc_ds.nc`
- `historic` saves and loads the historic soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_historic_soc_ds.nc`

The `save_instance_state` saves all non dataframe/dataset variables in a pickle file. This needs to be run to include all other set variable states. Without them many methods will not work.  



In [ ]:
# The commands in this cell saves the current state of the SoilData instance
FAI_soil.save_inventory()
FAI_soil.save_instance_state()

In [ ]:
# The commands in this cell instantiates and sets all the variable states of the instance to what it was when it was previously saved
FAI_soil = cm.SoilData.load_instance_state('FAI_soil')
FAI_soil.load_inventory()

In [ ]:
FAI_soil.check_attributes_status()

# Code development section below

New code below:

Code to check instance attributes below: